<a href="https://colab.research.google.com/github/Priya-on-loop/Bharatnatyam_mudras_recognition/blob/main/Computer_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 2: IMPORT ALL LIBRARIES
# ============================================================

# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import io
import json
import time
import os
import shutil
from pathlib import Path
from collections import Counter

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers, losses
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, EfficientNetB0, VGG16, InceptionV3,
    DenseNet121, Xception
)
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder

# Augmentation
import albumentations as A

# Explainability
from pytorch_grad_cam import GradCAM  # We'll use tf-keras Grad-CAM instead

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("✅ All libraries imported!")

In [ ]:
# ============================================================
# CELL 3: DATA LOADING - Multiple Methods for CV
# ============================================================

print("="*70)
print("📸 LOADING IMAGE DATA")
print("="*70)

# ──────────────────────────────────────────────
# METHOD 1: Built-in Dataset (Quick Demo)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("METHOD 1: Built-in Dataset (CIFAR-10)")
print("─"*50)

(X_train_cifar, y_train_cifar), (X_test_cifar, y_test_cifar) = keras.datasets.cifar10.load_data()
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train: {X_train_cifar.shape}, Test: {X_test_cifar.shape}")
print(f"Classes: {cifar_classes}")
print(f"Image size: {X_train_cifar.shape[1]}x{X_train_cifar.shape[2]}x{X_train_cifar.shape[3]}")

# ──────────────────────────────────────────────
# METHOD 2: TensorFlow Datasets (Flowers)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("METHOD 2: TensorFlow Flowers Dataset")
print("─"*50)

import tensorflow_datasets as tfds

(ds_train, ds_val, ds_test), ds_info = tfds.load(
    'tf_flowers',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    as_supervised=True,
    with_info=True
)

flower_classes = ds_info.features['label'].names
num_flower_classes = ds_info.features['label'].num_classes
print(f"Classes: {flower_classes}")
print(f"Num classes: {num_flower_classes}")
print(f"Train samples: {ds_info.splits['train'].num_examples}")

# ──────────────────────────────────────────────
# METHOD 3: From Directory Structure (Most Common in Tests!)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("METHOD 3: From Directory Structure")
print("─"*50)

# Create a demo directory structure from CIFAR-10
# In a real test, you'd have: data/train/cat/, data/train/dog/, etc.
base_dir = 'cv_dataset'
for split in ['train', 'val', 'test']:
    for cls in cifar_classes[:5]:  # Use 5 classes for speed
        os.makedirs(f'{base_dir}/{split}/{cls}', exist_ok=True)

# Populate with CIFAR images (demo)
from PIL import Image as PILImage

for i in range(500):  # 500 train images
    cls_idx = y_train_cifar[i][0]
    if cls_idx < 5:
        img = PILImage.fromarray(X_train_cifar[i])
        img.save(f'{base_dir}/train/{cifar_classes[cls_idx]}/img_{i}.png')

for i in range(100):  # 100 val images
    idx = i + 500
    cls_idx = y_train_cifar[idx][0]
    if cls_idx < 5:
        img = PILImage.fromarray(X_train_cifar[idx])
        img.save(f'{base_dir}/val/{cifar_classes[cls_idx]}/img_{idx}.png')

for i in range(100):  # 100 test images
    cls_idx = y_test_cifar[i][0]
    if cls_idx < 5:
        img = PILImage.fromarray(X_test_cifar[i])
        img.save(f'{base_dir}/test/{cifar_classes[cls_idx]}/img_{i}.png')

print(f"Directory structure created:")
for split in ['train', 'val', 'test']:
    for cls in sorted(os.listdir(f'{base_dir}/{split}')):
        n = len(os.listdir(f'{base_dir}/{split}/{cls}'))
        print(f"  {split}/{cls}: {n} images")

# ──────────────────────────────────────────────
# METHOD 4: From CSV + Image Folder (Kaggle Style)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("METHOD 4: CSV + Image Folder (Kaggle Style)")
print("─"*50)

print("""
# Typical Kaggle structure:
# train.csv → filename, label
# train_images/ → img_001.jpg, img_002.jpg, ...
#
# Loading code:
# df = pd.read_csv('train.csv')
# df['filepath'] = 'train_images/' + df['filename']
#
# Then use tf.data or ImageDataGenerator to load
""")

print("✅ All data loading methods demonstrated!")

In [ ]:
# ============================================================
# CELL 4: IMAGE EDA
# ============================================================

print("="*70)
print("🔍 IMAGE EXPLORATORY DATA ANALYSIS")
print("="*70)

# ──────────────────────────────────────────────
# 4.1: Class Distribution
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ CLASS DISTRIBUTION")
print("─"*50)

class_counts = {}
for cls in sorted(os.listdir(f'{base_dir}/train')):
    class_counts[cls] = len(os.listdir(f'{base_dir}/train/{cls}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(class_counts.keys(), class_counts.values(),
            color='steelblue', edgecolor='black')
axes[0].set_title('Training Class Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(class_counts.values(), labels=class_counts.keys(),
            autopct='%1.1f%%', colors=plt.cm.Set3.colors)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────
# 4.2: Sample Images
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("2️⃣ SAMPLE IMAGES PER CLASS")
print("─"*50)

fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for i, cls in enumerate(sorted(os.listdir(f'{base_dir}/train'))):
    img_files = os.listdir(f'{base_dir}/train/{cls}')[:5]
    for j, img_file in enumerate(img_files):
        img_path = f'{base_dir}/train/{cls}/{img_file}'
        img = plt.imread(img_path)
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(cls, fontsize=12, fontweight='bold')

plt.suptitle('Sample Images per Class', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────
# 4.3: Image Size Analysis
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("3️⃣ IMAGE SIZE ANALYSIS")
print("─"*50)

widths, heights, channels = [], [], []
for split in ['train']:
    for cls in os.listdir(f'{base_dir}/{split}'):
        for img_file in os.listdir(f'{base_dir}/{split}/{cls}')[:20]:
            img_path = f'{base_dir}/{split}/{cls}/{img_file}'
            img = cv2.imread(img_path)
            if img is not None:
                h, w, c = img.shape
                widths.append(w)
                heights.append(h)
                channels.append(c)

print(f"Width  → min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height → min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"Channels: {set(channels)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Image Widths')
axes[1].hist(heights, bins=20, color='coral', edgecolor='black')
axes[1].set_title('Image Heights')
plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────
# 4.4: Color Channel Analysis
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("4️⃣ COLOR CHANNEL ANALYSIS")
print("─"*50)

sample_imgs = []
for cls in os.listdir(f'{base_dir}/train'):
    img_files = os.listdir(f'{base_dir}/train/{cls}')[:5]
    for f in img_files:
        img = cv2.imread(f'{base_dir}/train/{cls}/{f}')
        if img is not None:
            sample_imgs.append(img)

sample_imgs = np.array(sample_imgs)
print(f"Mean pixel value: {sample_imgs.mean():.2f}")
print(f"Std pixel value: {sample_imgs.std():.2f}")
print(f"Min: {sample_imgs.min()}, Max: {sample_imgs.max()}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['blue', 'green', 'red']
for i, color in enumerate(colors):
    axes[i].hist(sample_imgs[:, :, :, i].flatten(), bins=50,
                 color=color, alpha=0.6, edgecolor='black')
    axes[i].set_title(f'{color.capitalize()} Channel')
    axes[i].set_xlabel('Pixel Value')
plt.suptitle('Color Channel Distributions', fontsize=14)
plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────
# 4.5: Brightness Analysis
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("5️⃣ BRIGHTNESS ANALYSIS")
print("─"*50)

brightness = [img.mean() for img in sample_imgs]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(brightness, bins=30, color='gold', edgecolor='black')
ax.set_title('Image Brightness Distribution')
ax.set_xlabel('Mean Pixel Value')
ax.axvline(np.mean(brightness), color='red', linestyle='--',
           label=f'Mean: {np.mean(brightness):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

print("\n✅ Image EDA Complete!")

In [ ]:
# ============================================================
# CELL 5: DATA PREPROCESSING & AUGMENTATION
# ============================================================

print("="*70)
print("⚙️ DATA PREPROCESSING & AUGMENTATION")
print("="*70)

IMG_SIZE = 128  # Resize all images to this
BATCH_SIZE = 32
NUM_CLASSES = 5

# ──────────────────────────────────────────────
# 5.1: Using tf.keras.utils.image_dataset_from_directory
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ LOADING WITH image_dataset_from_directory")
print("─"*50)

train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{base_dir}/train',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{base_dir}/val',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f'{base_dir}/test',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    seed=42
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")

# Optimize performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

# ──────────────────────────────────────────────
# 5.2: Data Augmentation Pipeline (Keras)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("2️⃣ DATA AUGMENTATION (Keras Layers)")
print("─"*50)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name="augmentation")

print("Augmentation layers:")
for layer in data_augmentation.layers:
    print(f"  • {layer.name}")

# ──────────────────────────────────────────────
# 5.3: Visualize Augmented Images
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("3️⃣ VISUALIZE AUGMENTATIONS")
print("─"*50)

for images, labels in train_ds.take(1):
    sample_image = images[0:1]  # Take one image

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i in range(3):
    for j in range(4):
        if i == 0 and j == 0:
            augmented = sample_image[0]
            axes[i, j].set_title("Original", fontweight='bold', color='red')
        else:
            augmented = data_augmentation(sample_image, training=True)[0]
        axes[i, j].imshow(augmented.numpy().astype('uint8'))
        axes[i, j].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=16)
plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────
# 5.4: Albumentations Pipeline (Advanced)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("4️⃣ ALBUMENTATIONS PIPELINE (Advanced)")
print("─"*50)

albumentations_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20,
                    min_holes=2, min_height=5, min_width=5, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

print("Albumentations transforms:")
for t in albumentations_transform.transforms:
    print(f"  • {t.__class__.__name__}")

# ──────────────────────────────────────────────
# 5.5: ImageDataGenerator (Legacy but Common)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("5️⃣ ImageDataGenerator (Legacy)")
print("─"*50)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    f'{base_dir}/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    f'{base_dir}/val',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print(f"Train batches: {len(train_generator)}")
print(f"Val batches: {len(val_generator)}")
print(f"Class indices: {train_generator.class_indices}")

# ──────────────────────────────────────────────
# 5.6: Normalization Comparison
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("6️⃣ NORMALIZATION METHODS")
print("─"*50)

normalization_methods = {
    'Rescale [0,1]': lambda x: x / 255.0,
    'Rescale [-1,1]': lambda x: (x / 127.5) - 1.0,
    'ImageNet Mean/Std': lambda x: (x/255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225],
    'Per-image': lambda x: (x - x.mean()) / (x.std() + 1e-7)
}

sample = X_train_cifar[0].astype(np.float32)
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes[0].imshow(sample.astype('uint8'))
axes[0].set_title('Original')
axes[0].axis('off')

for i, (name, func) in enumerate(normalization_methods.items()):
    normalized = func(sample.copy())
    # Clip for display
    display_img = np.clip(normalized, 0, 1) if 'Rescale [0,1]' in name else \
                  np.clip((normalized + 1) / 2, 0, 1) if '[-1,1]' in name else \
                  np.clip(normalized * 0.5 + 0.5, 0, 1)
    axes[i+1].imshow(display_img)
    axes[i+1].set_title(name, fontsize=9)
    axes[i+1].axis('off')
    print(f"  {name:25s} → range: [{normalized.min():.2f}, {normalized.max():.2f}]")

plt.tight_layout()
plt.show()

print("\n✅ Preprocessing & Augmentation Complete!")

In [ ]:
# ============================================================
# CELL 6: CNN FROM SCRATCH
# ============================================================

print("="*70)
print("🏗️ CNN FROM SCRATCH")
print("="*70)

# ──────────────────────────────────────────────
# 6.1: Build CNN Architecture
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ BUILDING CNN ARCHITECTURE")
print("─"*50)

def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """Build a custom CNN from scratch"""

    model = models.Sequential([
        # Input & Augmentation
        layers.Input(shape=input_shape),
        data_augmentation,

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),

        # Classifier
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

cnn_model = build_cnn()
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

# ──────────────────────────────────────────────
# 6.2: Callbacks
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("2️⃣ CALLBACKS")
print("─"*50)

cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.2,
        patience=5, min_lr=1e-7, verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_cnn.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print("Callbacks configured:")
print("  • EarlyStopping (patience=10)")
print("  • ReduceLROnPlateau (factor=0.2)")
print("  • ModelCheckpoint (best val_accuracy)")

# ──────────────────────────────────────────────
# 6.3: Train CNN
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("3️⃣ TRAINING CNN")
print("─"*50)

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=cnn_callbacks,
    verbose=1
)

# ──────────────────────────────────────────────
# 6.4: Training History
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("4️⃣ TRAINING HISTORY")
print("─"*50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(cnn_history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(cnn_history.history['val_accuracy'], label='Val', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(cnn_history.history['loss'], label='Train', linewidth=2)
axes[1].plot(cnn_history.history['val_loss'], label='Val', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN Training History', fontsize=16)
plt.tight_layout()
plt.show()

print(f"Best Val Accuracy: {max(cnn_history.history['val_accuracy']):.4f}")
print(f"Best Val Loss: {min(cnn_history.history['val_loss']):.4f}")

print("\n✅ CNN from Scratch Complete!")

In [ ]:
# ============================================================
# CELL 7: TRANSFER LEARNING (Multiple Architectures)
# ============================================================

print("="*70)
print("🚀 TRANSFER LEARNING")
print("="*70)

# ──────────────────────────────────────────────
# 7.1: Build Transfer Learning Models
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ BUILDING TRANSFER LEARNING MODELS")
print("─"*50)

def build_transfer_model(base_model_class, base_name, input_shape=(IMG_SIZE, IMG_SIZE, 3),
                         num_classes=NUM_CLASSES, fine_tune_layers=20):
    """Build a transfer learning model"""

    # Load base model (pretrained on ImageNet)
    base_model = base_model_class(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )

    # Freeze base model
    base_model.trainable = False

    # Build full model
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Preprocessing for specific architecture
    if 'MobileNet' in base_name:
        x = mobilenet_preprocess(x)
    elif 'ResNet' in base_name:
        x = resnet_preprocess(x)
    elif 'Efficient' in base_name:
        x = efficientnet_preprocess(x)
    else:
        x = x / 255.0

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=f'TL_{base_name}')

    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, base_model

# Define architectures to try
architectures = {
    'MobileNetV2': MobileNetV2,
    'EfficientNetB0': EfficientNetB0,
    'ResNet50': ResNet50,
}

tl_models = {}
tl_histories = {}

# ──────────────────────────────────────────────
# 7.2: Train Each Architecture
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("2️⃣ TRAINING TRANSFER LEARNING MODELS")
print("─"*50)

for arch_name, arch_class in architectures.items():
    print(f"\n{'='*50}")
    print(f"Training: {arch_name}")
    print(f"{'='*50}")

    model, base_model = build_transfer_model(
        arch_class, arch_name,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        num_classes=NUM_CLASSES
    )

    print(f"Total params: {model.count_params():,}")
    print(f"Trainable params: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

    tl_callbacks = [
        callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2,
                                     patience=4, min_lr=1e-7, verbose=0),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        callbacks=tl_callbacks,
        verbose=1
    )

    tl_models[arch_name] = {'model': model, 'base_model': base_model}
    tl_histories[arch_name] = history

    val_acc = max(history.history['val_accuracy'])
    print(f"\n✅ {arch_name} Best Val Accuracy: {val_acc:.4f}")

# ──────────────────────────────────────────────
# 7.3: Fine-Tuning (Unfreeze Top Layers)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("3️⃣ FINE-TUNING (Unfreeze Top Layers)")
print("─"*50)

# Pick best model for fine-tuning
best_arch = max(tl_histories.keys(),
                key=lambda k: max(tl_histories[k].history['val_accuracy']))
print(f"Fine-tuning best model: {best_arch}")

best_model = tl_models[best_arch]['model']
best_base = tl_models[best_arch]['base_model']

# Unfreeze top layers
best_base.trainable = True
fine_tune_at = len(best_base.layers) - 20  # Unfreeze last 20 layers

for layer in best_base.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total layers: {len(best_base.layers)}")
print(f"Unfrozen layers: {len(best_base.layers) - fine_tune_at}")
print(f"Trainable params: {sum([tf.keras.backend.count_params(w) for w in best_model.trainable_weights]):,}")

# Recompile with lower learning rate
best_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),  # Lower LR for fine-tuning!
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True, verbose=0),
]

print("\nStarting fine-tuning...")
ft_history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=fine_tune_callbacks,
    verbose=1
)

print(f"\n✅ Fine-tuned Val Accuracy: {max(ft_history.history['val_accuracy']):.4f}")

# ──────────────────────────────────────────────
# 7.4: Compare All Models
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("4️⃣ MODEL COMPARISON")
print("─"*50)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for arch_name, history in tl_histories.items():
    axes[0].plot(history.history['val_accuracy'], label=arch_name, linewidth=2)
    axes[1].plot(history.history['val_loss'], label=arch_name, linewidth=2)

# Add fine-tuning history
ft_start = len(tl_histories[best_arch].history['val_accuracy'])
axes[0].plot(range(ft_start, ft_start + len(ft_history.history['val_accuracy'])),
             ft_history.history['val_accuracy'],
             label=f'{best_arch} (Fine-tuned)', linewidth=2, linestyle='--')

axes[0].set_title('Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Transfer Learning Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
print("\n📊 Final Results:")
print(f"{'Model':<25} {'Best Val Acc':>12} {'Best Val Loss':>14}")
print("─" * 55)
for arch_name, history in tl_histories.items():
    acc = max(history.history['val_accuracy'])
    loss = min(history.history['val_loss'])
    print(f"{arch_name:<25} {acc:>12.4f} {loss:>14.4f}")
ft_acc = max(ft_history.history['val_accuracy'])
ft_loss = min(ft_history.history['val_loss'])
print(f"{best_arch + ' (FT)':<25} {ft_acc:>12.4f} {ft_loss:>14.4f}")

print("\n✅ Transfer Learning Complete!")

In [ ]:
# ============================================================
# CELL 8: COMPREHENSIVE MODEL EVALUATION
# ============================================================

print("="*70)
print("📊 MODEL EVALUATION")
print("="*70)

# Use the fine-tuned model
eval_model = best_model

# ──────────────────────────────────────────────
# 8.1: Test Set Evaluation
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ TEST SET EVALUATION")
print("─"*50)

test_loss, test_acc = eval_model.evaluate(test_ds, verbose=1)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# ──────────────────────────────────────────────
# 8.2: Predictions & Classification Report
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("2️⃣ CLASSIFICATION REPORT")
print("─"*50)

# Collect all predictions
y_true = []
y_pred = []
y_prob = []

for images, labels in test_ds:
    preds = eval_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print(classification_report(y_true, y_pred, target_names=class_names))

# ──────────────────────────────────────────────
# 8.3: Confusion Matrix
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("3️⃣ CONFUSION MATRIX")
print("─"*50)

cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────
# 8.4: Misclassified Images
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("4️⃣ MISCLASSIFIED IMAGES")
print("─"*50)

misclassified_idx = np.where(y_true != y_pred)[0]
print(f"Total misclassified: {len(misclassified_idx)} / {len(y_true)}")

if len(misclassified_idx) > 0:
    n_show = min(12, len(misclassified_idx))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()

    # Get test images
    test_images = []
    test_labels = []
    for images, labels in test_ds:
        test_images.extend(images.numpy())
        test_labels.extend(np.argmax(labels.numpy(), axis=1))
    test_images = np.array(test_images)

    for i in range(n_show):
        idx = misclassified_idx[i]
        axes[i].imshow(test_images[idx].astype('uint8'))
        axes[i].set_title(
            f"True: {class_names[y_true[idx]]}\n"
            f"Pred: {class_names[y_pred[idx]]}\n"
            f"Conf: {y_prob[idx][y_pred[idx]]:.1%}",
            color='red', fontsize=9
        )
        axes[i].axis('off')

    for j in range(n_show, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Misclassified Examples', fontsize=16)
    plt.tight_layout()
    plt.show()

# ──────────────────────────────────────────────
# 8.5: Prediction Confidence Distribution
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("5️⃣ PREDICTION CONFIDENCE")
print("─"*50)

max_probs = np.max(y_prob, axis=1)
correct_mask = y_true == y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(max_probs[correct_mask], bins=30, alpha=0.7,
             label='Correct', color='green', edgecolor='black')
axes[0].hist(max_probs[~correct_mask], bins=30, alpha=0.7,
             label='Incorrect', color='red', edgecolor='black')
axes[0].set_title('Prediction Confidence')
axes[0].set_xlabel('Max Probability')
axes[0].legend()

# Confidence vs Accuracy
bins = np.linspace(0, 1, 11)
bin_acc = []
for i in range(len(bins)-1):
    mask = (max_probs >= bins[i]) & (max_probs < bins[i+1])
    if mask.sum() > 0:
        bin_acc.append(correct_mask[mask].mean())
    else:
        bin_acc.append(0)

axes[1].bar(range(len(bin_acc)), bin_acc, color='steelblue', edgecolor='black')
axes[1].set_xticks(range(len(bin_acc)))
axes[1].set_xticklabels([f'{bins[i]:.1f}-{bins[i+1]:.1f}' for i in range(len(bins)-1)],
                         rotation=45, ha='right')
axes[1].set_title('Accuracy by Confidence Bin')
axes[1].set_ylabel('Accuracy')
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n✅ Model Evaluation Complete!")

In [ ]:
# ============================================================
# CELL 9: GRAD-CAM (Visual Explainability)
# ============================================================

print("="*70)
print("🔍 GRAD-CAM VISUAL EXPLAINABILITY")
print("="*70)

# ──────────────────────────────────────────────
# 9.1: Grad-CAM Implementation
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ GRAD-CAM IMPLEMENTATION")
print("─"*50)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap"""

    # Create model that maps input to last conv layer output + predictions
    grad_model = keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Record operations for gradient computation
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # Gradient of the predicted class w.r.t. conv layer output
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # Global average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the channels by gradient importance
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # ReLU + normalize
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy(), pred_index.numpy()

def display_gradcam(img, heatmap, alpha=0.4):
    """Overlay heatmap on image"""
    heatmap = np.uint8(255 * heatmap)
    jet = plt.cm.jet
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = keras.utils.img_to_array(jet_heatmap)

    superimposed = jet_heatmap * alpha + img
    superimposed = keras.utils.array_to_img(superimposed)
    return superimposed

# ──────────────────────────────────────────────
# 9.2: Find Last Conv Layer
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("2️⃣ FINDING LAST CONV LAYER")
print("─"*50)

# Find the last convolutional layer
last_conv_layer = None
for layer in reversed(eval_model.layers):
    if isinstance(layer, (keras.layers.Conv2D, keras.layers.SeparableConv2D)):
        last_conv_layer = layer.name
        break

if not last_conv_layer:
    # For transfer learning models, search inside the base model
    for layer in reversed(eval_model.layers):
        if hasattr(layer, 'layers'):
            for sub_layer in reversed(layer.layers):
                if isinstance(sub_layer, (keras.layers.Conv2D,
                                          keras.layers.SeparableConv2D)):
                    last_conv_layer = sub_layer.name
                    break
        if last_conv_layer:
            break

print(f"Last conv layer: {last_conv_layer}")

# ──────────────────────────────────────────────
# 9.3: Generate Grad-CAM for Test Images
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("3️⃣ GRAD-CAM VISUALIZATIONS")
print("─"*50)

# Get some test images
test_images_list = []
test_labels_list = []
for images, labels in test_ds.take(2):
    test_images_list.extend(images.numpy())
    test_labels_list.extend(np.argmax(labels.numpy(), axis=1))

n_show = min(8, len(test_images_list))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))

if n_show == 1:
    axes = axes.reshape(1, -1)

for i in range(n_show):
    img = test_images_list[i]
    true_label = test_labels_list[i]

    # Prepare input
    img_array = np.expand_dims(img, axis=0).astype(np.float32)
    if img_array.max() > 1:
        img_array = img_array / 255.0

    try:
        # Generate heatmap
        heatmap, pred_idx = make_gradcam_heatmap(
            img_array, eval_model, last_conv_layer
        )

        # Display
        axes[i, 0].imshow(img.astype('uint8'))
        axes[i, 0].set_title(f'Original\nTrue: {class_names[true_label]}')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(heatmap, cmap='jet')
        axes[i, 1].set_title('Grad-CAM Heatmap')
        axes[i, 1].axis('off')

        overlay = display_gradcam(img.astype('uint8'), heatmap)
        axes[i, 2].imshow(overlay)
        pred_label = class_names[pred_idx]
        color = 'green' if pred_idx == true_label else 'red'
        axes[i, 2].set_title(f'Overlay\nPred: {pred_label}', color=color)
        axes[i, 2].axis('off')

    except Exception as e:
        axes[i, 0].imshow(img.astype('uint8'))
        axes[i, 0].set_title(f'Error: {str(e)[:30]}')
        axes[i, 0].axis('off')
        axes[i, 1].axis('off')
        axes[i, 2].axis('off')

plt.suptitle('Grad-CAM Visualizations', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\n✅ Grad-CAM Complete!")

In [ ]:
# ============================================================
# CELL 10: OBJECT DETECTION BASICS (BONUS)
# ============================================================

print("="*70)
print("🎯 OBJECT DETECTION BASICS")
print("="*70)

# ──────────────────────────────────────────────
# 10.1: Pre-trained Object Detection (TensorFlow Hub)
# ──────────────────────────────────────────────
print("\n" + "─"*50)
print("1️⃣ PRE-TRAINED DETECTOR (TF Hub)")
print("─"*50)

import tensorflow_hub as hub

print("Loading SSD MobileNet V2 detector...")
detector = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2")

# COCO class labels (subset)
coco_labels = {
    1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
    6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
    16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant',
    21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 28: 'suitcase',
    44: 'bottle', 47: 'cup', 49: 'fork', 50: 'knife', 51: 'spoon',
    62: 'tv', 63: 'laptop', 64: 'mouse', 67: 'cell phone', 73: 'book',
    77: 'teddy bear', 78: 'hair drier', 79: 'toothbrush'
}

def detect_objects(image, threshold=0.3):
    """Run object detection on an image"""
    img_tensor = tf.convert_to_tensor(image, dtype=tf.uint8)
    img_tensor = tf.expand_dims(img_tensor, 0)

    result = detector(img_tensor)

    boxes = result['detection_boxes'][0].numpy()
    classes = result['detection_classes'][0].numpy().astype(int)
    scores = result['detection_scores'][0].numpy()

    detections = []
    for box, cls, score in zip(boxes, classes, scores):
        if score >= threshold:
            detections.append({
                'box': box,
                'class': coco_labels.get(cls, f'class_{cls}'),
                'score': score
            })

    return detections

def draw_detections(image, detections):
    """Draw bounding boxes on image"""
    img_copy = image.copy()
    h, w = img_copy.shape[:2]

    for det in detections:
        ymin, xmin, ymax, xmax = det['box']
        x1, y1 = int(xmin * w), int(ymin * h)
        x2, y2 = int(xmax * w), int(ymax * h)

        color = (0, 255, 0)
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, 2)
        label = f"{det['class']}: {det['score']:.0%}"
        cv2.putText(img_copy, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    return img_copy

# Test with a sample image (create a simple test image)
test_img = cv2.resize(X_train_cifar[0], (300, 300))
detections = detect_objects(test_img, threshold=0.2)

print(f"Detected {len(detections)} objects")
for det in detections[:5]:
    print(f"  {det['class']}: {det['score']:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(test_img)
axes[0].set_title('Original')
axes[0].axis('off')

if detections:
    result_img = draw_detections(test_img, detections)
    axes[1].imshow(result_img)
    axes[1].set_title(f'Detections ({len(detections)})')
else:
    axes[1].imshow(test_img)
    axes[1].set_title('No detections (low-res image)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("""
📌 OBJECT DETECTION APPROACHES FOR TESTS:
─────────────────────────────────────────
1. Pre-trained (TF Hub): Quick, no training needed
2. YOLOv8 (Ultralytics): !pip install ultralytics
3. Custom training: TensorFlow Object Detection API
4. Hugging Face: transformers pipeline("object-detection")
""")

print("✅ Object Detection Demo Complete!")

In [ ]:
# ============================================================
# CELL 11: SAVE MODEL & ARTIFACTS
# ============================================================

print("="*70)
print("💾 SAVING MODEL & ARTIFACTS")
print("="*70)

os.makedirs('cv_model_artifacts', exist_ok=True)

# ---- 11.1: Save Keras Model ----
eval_model.save('cv_model_artifacts/cv_classifier.keras')
print("✅ Saved Keras model (.keras)")

# Also save in SavedModel format
eval_model.save('cv_model_artifacts/cv_classifier_savedmodel')
print("✅ Saved SavedModel format")

# ---- 11.2: Save TFLite Model (for mobile/edge) ----
converter = tf.lite.TFLiteConverter.from_keras_model(eval_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('cv_model_artifacts/cv_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print("✅ Saved TFLite model")

# ---- 11.3: Save Config ----
cv_config = {
    'model_architecture': best_arch,
    'input_size': [IMG_SIZE, IMG_SIZE, 3],
    'num_classes': NUM_CLASSES,
    'class_names': class_names,
    'test_accuracy': float(test_acc),
    'fine_tuned': True,
    'normalization': 'mobilenet_preprocess' if 'Mobile' in best_arch else 'imagenet'
}

with open('cv_model_artifacts/cv_config.json', 'w') as f:
    json.dump(cv_config, f, indent=2)
print("✅ Saved config")

# ---- 11.4: Verify ----
print("\n" + "─"*50)
print("VERIFICATION")
print("─"*50)

loaded_model = keras.models.load_model('cv_model_artifacts/cv_classifier.keras')
test_loss_v, test_acc_v = loaded_model.evaluate(test_ds, verbose=0)
print(f"Loaded model test accuracy: {test_acc_v:.4f}")

for f in os.listdir('cv_model_artifacts'):
    path = f'cv_model_artifacts/{f}'
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  📁 {f} ({size:.2f} MB)")

print("\n✅ All CV artifacts saved!")

In [ ]:
# ============================================================
# CELL 12: STREAMLIT WEB APP FOR CV
# ============================================================

print("="*70)
print("🚀 BUILDING STREAMLIT CV APP")
print("="*70)

!pip install streamlit -q

%%writefile cv_app.py

import streamlit as st
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image
import json
import cv2
import io
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────
# PAGE CONFIG
# ──────────────────────────────────────────────
st.set_page_config(
    page_title="🖼️ Image Classifier",
    page_icon="🖼️",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .main-header { font-size: 2.5rem; color: #1f77b4; text-align: center; }
    .upload-box {
        border: 3px dashed #1f77b4;
        border-radius: 15px;
        padding: 2rem;
        text-align: center;
        background-color: #f0f7ff;
    }
    .result-success {
        background: linear-gradient(135deg, #11998e, #38ef7d);
        padding: 1.5rem; border-radius: 10px; color: white; text-align: center;
    }
</style>
""", unsafe_allow_html=True)

# ──────────────────────────────────────────────
# LOAD MODEL
# ──────────────────────────────────────────────
@st.cache_resource
def load_cv_model():
    model = keras.models.load_model('cv_model_artifacts/cv_classifier.keras')
    with open('cv_model_artifacts/cv_config.json', 'r') as f:
        config = json.load(f)
    return model, config

try:
    model, config = load_cv_model()
    CLASS_NAMES = config['class_names']
    IMG_SIZE = tuple(config['input_size'][:2])
    MODEL_LOADED = True
except Exception as e:
    MODEL_LOADED = False
    st.error(f"Error: {e}")

# ──────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────
st.sidebar.markdown("## 🧭 Navigation")
page = st.sidebar.radio("Go to:", [
    "🏠 Home", "🔮 Classify Image", "📊 Batch Predict", "ℹ️ About"
])

st.sidebar.markdown("---")
if MODEL_LOADED:
    st.sidebar.success("✅ Model Loaded")
    st.sidebar.info(f"**Architecture:** {config.get('model_architecture', 'N/A')}")
    st.sidebar.info(f"**Classes:** {len(CLASS_NAMES)}")
    st.sidebar.info(f"**Input Size:** {IMG_SIZE}")

# ──────────────────────────────────────────────
# HELPER FUNCTIONS
# ──────────────────────────────────────────────
def preprocess_image(image, target_size):
    """Preprocess uploaded image for model"""
    img = image.resize(target_size)
    img_array = np.array(img, dtype=np.float32)
    if img_array.shape[-1] == 4:  # RGBA → RGB
        img_array = img_array[:, :, :3]
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array

def predict_image(image, model, class_names, target_size):
    """Run prediction on image"""
    processed = preprocess_image(image, target_size)
    predictions = model.predict(processed, verbose=0)[0]
    top_idx = np.argsort(predictions)[::-1]
    results = []
    for idx in top_idx:
        results.append({
            'class': class_names[idx],
            'confidence': float(predictions[idx])
        })
    return results, predictions

# ──────────────────────────────────────────────
# PAGE: HOME
# ──────────────────────────────────────────────
if page == "🏠 Home":
    st.markdown('<p class="main-header">🖼️ Image Classification App</p>',
                unsafe_allow_html=True)
    st.markdown("<p style='text-align:center; color:#666; font-size:1.2rem;'>"
                "Upload an image and get instant AI-powered classification</p>",
                unsafe_allow_html=True)

    st.markdown("---")

    c1, c2, c3 = st.columns(3)
    c1.metric("🏗️ Architecture", config.get('model_architecture', 'N/A') if MODEL_LOADED else '-')
    c2.metric("📦 Classes", len(CLASS_NAMES) if MODEL_LOADED else '-')
    c3.metric("🎯 Test Accuracy",
              f"{config.get('test_accuracy', 0):.1%}" if MODEL_LOADED else '-')

    st.markdown("### 📌 How to Use")
    st.markdown("""
    1. Go to **Classify Image** page
    2. Upload an image (JPG, PNG, WEBP)
    3. View the prediction with confidence scores
    4. Try **Batch Predict** for multiple images
    """)

# ──────────────────────────────────────────────
# PAGE: CLASSIFY IMAGE
# ──────────────────────────────────────────────
elif page == "🔮 Classify Image":
    st.markdown("## 🔮 Image Classification")

    if not MODEL_LOADED:
        st.error("Model not loaded.")
        st.stop()

    col_left, col_right = st.columns([1, 1])

    with col_left:
        st.markdown("### 📤 Upload Image")
        uploaded_file = st.file_uploader(
            "Choose an image...",
            type=['jpg', 'jpeg', 'png', 'webp', 'bmp'],
            help="Upload any image to classify"
        )

        # Camera input option
        camera_image = st.camera_input("📷 Or take a photo")

        image_source = uploaded_file or camera_image

    with col_right:
        if image_source is not None:
            image = Image.open(image_source).convert('RGB')
            st.image(image, caption="Uploaded Image", use_container_width=True)

            if st.button("🔍 Classify", type="primary", use_container_width=True):
                with st.spinner("Analyzing image..."):
                    results, raw_preds = predict_image(
                        image, model, CLASS_NAMES, IMG_SIZE
                    )

                st.markdown("---")
                st.markdown("### 📋 Results")

                # Top prediction
                top = results[0]
                st.markdown(f"""
                <div class="result-success">
                    <h2>🏷️ {top['class'].upper()}</h2>
                    <h3>Confidence: {top['confidence']:.1%}</h3>
                </div>
                """, unsafe_allow_html=True)

                # All predictions
                st.markdown("#### All Predictions")
                for r in results:
                    st.progress(r['confidence'])
                    st.text(f"{r['class']}: {r['confidence']:.2%}")

                # Bar chart
                fig, ax = plt.subplots(figsize=(8, 4))
                classes = [r['class'] for r in results]
                confs = [r['confidence'] for r in results]
                colors = ['#28a745' if i == 0 else '#6c757d'
                          for i in range(len(classes))]
                ax.barh(classes[::-1], confs[::-1], color=colors[::-1])
                ax.set_xlim(0, 1)
                ax.set_xlabel('Confidence')
                ax.set_title('Prediction Probabilities')
                st.pyplot(fig)
                plt.close()
        else:
            st.info("👈 Upload an image to get started")

# ──────────────────────────────────────────────
# PAGE: BATCH PREDICT
# ──────────────────────────────────────────────
elif page == "📊 Batch Predict":
    st.markdown("## 📊 Batch Prediction")

    if not MODEL_LOADED:
        st.error("Model not loaded.")
        st.stop()

    uploaded_files = st.file_uploader(
        "Upload multiple images...",
        type=['jpg', 'jpeg', 'png', 'webp'],
        accept_multiple_files=True
    )

    if uploaded_files:
        st.markdown(f"### {len(uploaded_files)} images uploaded")

        if st.button("🔍 Classify All", type="primary"):
            results_data = []

            cols = st.columns(min(4, len(uploaded_files)))

            for i, file in enumerate(uploaded_files):
                image = Image.open(file).convert('RGB')
                results, _ = predict_image(image, model, CLASS_NAMES, IMG_SIZE)
                top = results[0]

                results_data.append({
                    'filename': file.name,
                    'prediction': top['class'],
                    'confidence': f"{top['confidence']:.2%}"
                })

                with cols[i % len(cols)]:
                    st.image(image, caption=f"{top['class']}\n{top['confidence']:.1%}",
                            use_container_width=True)

            # Results table
            st.markdown("### 📋 Summary Table")
            st.dataframe(
                pd.DataFrame(results_data),
                use_container_width=True,
                hide_index=True
            )

            # Download results
            csv = pd.DataFrame(results_data).to_csv(index=False)
            st.download_button(
                "📥 Download Results CSV",
                csv, "predictions.csv", "text/csv"
            )

# ──────────────────────────────────────────────
# PAGE: ABOUT
# ──────────────────────────────────────────────
elif page == "ℹ️ About":
    st.markdown("## ℹ️ About This CV Application")

    st.markdown(f"""
    ### 🛠️ Tech Stack
    | Component | Technology |
    |-----------|-----------|
    | **Deep Learning** | TensorFlow {tf.__version__} |
    | **Architecture** | {config.get('model_architecture', 'N/A')} |
    | **Web Framework** | Streamlit |
    | **Image Processing** | OpenCV, Pillow |
    | **Deployment** | Streamlit + LocalTunnel |

    ### 🔄 CV Pipeline Steps
    1. ✅ Image Data Loading (Directory, TFDS, CSV)
    2. ✅ Image EDA (Distributions, Color Analysis)
    3. ✅ Data Augmentation (Keras + Albumentations)
    4. ✅ CNN from Scratch
    5. ✅ Transfer Learning (MobileNet, EfficientNet, ResNet)
    6. ✅ Fine-Tuning (Unfreeze top layers)
    7. ✅ Evaluation (CM, Report, Misclassifications)
    8. ✅ Grad-CAM Explainability
    9. ✅ Model Export (Keras, TFLite, SavedModel)
    10. ✅ Web Deployment (Streamlit)

    ### 📊 Model Details
    - **Input Size:** {IMG_SIZE}
    - **Classes:** {', '.join(CLASS_NAMES)}
    - **Test Accuracy:** {config.get('test_accuracy', 0):.2%}
    """)

In [ ]:
# ============================================================
# CELL 13: LAUNCH STREAMLIT CV APP
# ============================================================

print("="*70)
print("🚀 LAUNCHING CV STREAMLIT APP")
print("="*70)

!pkill -f streamlit 2>/dev/null || true
!pkill -f lt 2>/dev/null || true

import threading
import time
import re

def run_streamlit():
    !streamlit run cv_app.py --server.port 8501 \
        --server.headless true \
        --server.enableCORS false \
        --server.enableXsrfProtection false \
        --server.maxUploadSize 50 \
        --browser.gatherUsageStats false &

def run_tunnel():
    !lt --port 8501 --subdomain cv-demo-app 2>&1 | tee tunnel.log &

print("⏳ Starting Streamlit...")
threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(6)

print("⏳ Creating tunnel...")
threading.Thread(target=run_tunnel, daemon=True).start()
time.sleep(5)

try:
    with open('tunnel.log', 'r') as f:
        url_match = re.search(r'(https://[^\s]+)', f.read())
    public_url = url_match.group(1) if url_match else "https://cv-demo-app.loca.lt"
except:
    public_url = "https://cv-demo-app.loca.lt"

print(f"""
┌─────────────────────────────────────────────────────┐
│                                                     │
│   🌐 Public URL:  {public_url:<35s} │
│   📍 Local URL:   http://localhost:8501             │
│                                                     │
│   ⚠️  Click "Click to Continue" on the tunnel page  │
│                                                     │
│   📱 App Features:                                  │
│     🏠 Home          - Model overview               │
│     🔮 Classify      - Upload & predict single      │
│     📊 Batch         - Multiple images + CSV        │
│     ℹ️  About         - Pipeline documentation       │
│                                                     │
└─────────────────────────────────────────────────────┘
""")

print("✅ CV Streamlit app is running!")